# Pydantic Fundamentals: Schemas an LLM Can Fill [Step 3 - Pydantic fundamentals]

> **MLCourse - Agentic AI - Output Parsers and Pydantic**

> Stage in the capstone: capstone answers are parsed into a strict AnswerWithSources
> schema so citations cannot hallucinate.

## What you'll learn
- what a schema is and why free-text model output breaks software without one
- declaring typed fields with `BaseModel` plus defaults, ranges, and descriptions via `Field`
- how validation succeeds silently and fails loudly through `ValidationError`
- composing nested models (`Address` inside `Customer`)
- adding custom business rules with `@field_validator`
- round-tripping models through dicts and JSON strings

No language model is called anywhere in this notebook. We build the CONTRACT first;
the next notebook forces a real chat model to fill exactly this kind of shape.

In [1]:
# Standard first cell for every MLCourse notebook: imports, inline plotting,
# and automatic discovery of the track-level .env file.
import json                                 # stdlib JSON reader used in the round trip cell
import os                                   # read environment variables such as GROQ_API_KEY
from pathlib import Path                    # walk up the folder tree hunting for .env

try:                                        # Jupyter kernels define get_ipython();
    get_ipython().run_line_magic("matplotlib", "inline")  # render plots inside the notebook
except NameError:                           # plain python runs have no IPython,
    pass                                    # so skip the magic silently

from dotenv import load_dotenv              # loads KEY=VALUE lines into os.environ


def find_track_env(start: Path) -> "Path | None":
    """Climb from *start* upward until 03_agentic_ai/.env appears."""
    for folder in (start, *start.parents):             # current dir, then every parent
        candidate = folder / "03_agentic_ai" / ".env"  # track secrets live at this spot
        if candidate.is_file():                        # hit: stop climbing immediately
            return candidate
    return None                                        # miss everywhere: caller decides


_env_path = find_track_env(Path.cwd())     # search from wherever the kernel started
if _env_path is not None:                  # found the track root?
    load_dotenv(_env_path)                 # push GROQ_API_KEY etc. into os.environ
    print("[setup] loaded env:", _env_path)
else:
    print("[setup] no 03_agentic_ai/.env found - live demos will be skipped")

[setup] no 03_agentic_ai/.env found - live demos will be skipped


## 1. The problem a schema solves

Imagine asking a model: "Answer this question, cite sources, rate your confidence."
It replies with a friendly paragraph. Now write CODE against that paragraph: where
does the answer end? which URLs were cited? is confidence the word "high" or 0.87?
Every consumer of the reply (a database, a UI badge, an agent router) needs typed
values, not prose.

A pydantic model is a machine-checkable contract: field names, types, ranges, and
documentation in one class. Define it once and three things come free - validation
on the way IN, precise errors when data is bad, serialization on the way OUT.

> **Pro tip:** design the schema BEFORE writing the prompt. The field names and
> descriptions you pick here are later shipped to the model as part of its format
> instructions, so schema design IS prompt engineering.

In [2]:
from pydantic import BaseModel, Field       # the declarative base and the field builder


class AnswerDraft(BaseModel):               # our target shape for question answering
    """Schema we will later force an LLM to fill - an answer plus its receipts."""

    question: str = Field(description="The user question, repeated verbatim")            # required: no default
    answer: str = Field(description="The short factual answer")                          # required
    confidence: float = Field(ge=0.0, le=1.0, description="Self-rated certainty, 0 to 1")  # range-checked float
    sources: list[str] = Field(default_factory=list, max_length=5)   # default_factory avoids the shared-list bug


draft = AnswerDraft(                        # construction IS validation - bad data raises here
    question="Which gas do plants absorb for photosynthesis?",
    answer="Carbon dioxide",
    confidence=0.93,
    sources=["wikipedia.org/wiki/Photosynthesis"],
)
print(draft.answer, "/", draft.confidence)  # plain attribute access on fully typed fields
print(type(draft.confidence).__name__)      # 'float' - coercion already ran at creation

Carbon dioxide / 0.93
float


## 2. Validation: quiet success, loud failure

Constraints run automatically at construction time. Valid input just becomes an
object; invalid input raises ONE `ValidationError` listing EVERY broken field at
once - not merely the first offender. Its `.errors()` method returns machine-readable
dicts carrying the dotted path (`loc`), a stable error `type`, and a human `msg`.

> **Common pitfall:** catching bare `Exception` around model creation turns schema
> bugs into mystery crashes far downstream. Catch `ValidationError` specifically and
> decide what to do with each failed field.

In [3]:
from pydantic import ValidationError        # raised whenever any check fails

bad_input = {
    "question": "",                         # empty is allowed for now (custom rules come later)
    "answer": None,                         # wrong type entirely - None is not a str
    "confidence": 1.7,                      # violates le=1.0
    "sources": ["a"] * 9,                   # violates max_length=5
}

try:
    AnswerDraft(**bad_input)                # validation fires eagerly, right here
except ValidationError as exc:              # one exception, many findings inside
    print("error count:", exc.error_count())                 # total problems detected
    for err in exc.errors():                # iterate each individual finding
        path = ".".join(str(p) for p in err["loc"])          # nested fields join with dots
        print(f"  [{path}] {err['type']}: {err['msg']}")     # stable type + readable message
print("program continues - WE owned the bad data instead of crashing")

error count: 3
  [answer] string_type: Input should be a valid string
  [confidence] less_than_equal: Input should be less than or equal to 1
  [sources] too_long: List should have at most 5 items after validation, not 9
program continues - WE owned the bad data instead of crashing


## 3. Nested models: Address inside Customer

Real payloads nest: an order holds a customer, the customer holds an address, the
address holds a country. Pydantic composes the same way - a field whose type is
itself a `BaseModel`. Hand it a plain dict and pydantic coerces AND validates it
recursively, so a malformed inner value dies at the boundary, never deep in logic.

In [4]:
class Address(BaseModel):                   # building block, used INSIDE Customer
    """Postal address - deliberately small so the nesting stays obvious."""

    street: str = Field(description="Street name and house number")  # required
    city: str = Field(description="City name")                       # required
    country: str = Field(description="Country of residence")         # required


class Customer(BaseModel):                  # parent model owning a nested model
    """A customer record - the classic nested-schema example."""

    name: str = Field(description="Full display name")               # required string
    age: int = Field(ge=18, le=120, description="Age in years")      # adults-only range
    address: Address                                                 # nested model type


cust = Customer(                            # feed a dict for 'address': auto-coerced
    name="Ada Lovelace",
    age=36,
    address={"street": "12 St James's Sq", "city": "London", "country": "UK"},
)
print(cust.address.city)                     # attribute access straight THROUGH the nest
print(cust.model_dump()["address"]["city"])  # dump() recurses into plain nested dicts too

London
London


## 4. Custom rules with @field_validator

Types and ranges cannot say "country must be UPPERCASE". A field validator hooks
your own function onto one field; it receives the already-coerced value and returns
the cleaned version - or raises to reject. Perfect for normalization plus those last
business rules that declarative constraints cannot express.

> **Pro tip:** validators that RETURN a new value act as normalizers; validators
> that RAISE encode domain law. Keep each validator doing exactly one of the two jobs.

In [5]:
from pydantic import field_validator        # decorator registering the hook


class ShippingAddress(Address):             # inherit Address fields, ADD a stricter rule
    """An address whose country gets force-normalized to uppercase."""

    @field_validator("country")             # this rule guards ONLY the country field
    @classmethod
    def country_upper(cls, value: str) -> str:   # runs AFTER pydantic's type coercion
        trimmed = value.strip()             # tolerate sloppy surrounding whitespace
        if len(trimmed) < 2:                # domain rule: real country names exist
            raise ValueError("country needs at least 2 characters")
        return trimmed.upper()              # normalize: 'uk', 'Uk', ' us ' -> uppercase


ok_addr = ShippingAddress(street="1 Main St", city="Springfield", country=" us ")
print(ok_addr.country)                      # US - the validator rewrote the stored value

try:
    ShippingAddress(street="1 Main St", city="Springfield", country="X")
except ValidationError as exc:              # our ValueError became a structured error
    print("rejected:", exc.errors()[0]["msg"])   # note loc points at ('country',)

US
rejected: Value error, country needs at least 2 characters


## 5. Round trip: object -> dict -> JSON -> object

Storage layers and HTTP speak dicts and JSON strings; your code wants objects.
Pydantic ships both directions: `model_dump` / `model_dump_json` serialize, while
`model_validate` / `model_validate_json` parse AND revalidate in one call - so a
corrupted payload dies at the door instead of poisoning program state.

In [6]:
as_dict = cust.model_dump()                  # nested models become nested plain dicts
as_json = cust.model_dump_json(indent=2)     # canonical JSON text, ready to store or send
print(as_dict["address"]["country"])         # living in plain-dict land now
print(json.loads(as_json)["age"])            # stdlib json reads it happily

rebuilt = Customer.model_validate_json(as_json)   # parse + validate in one shot
assert rebuilt == cust                       # equality compares FIELD VALUES, not identity
print("round trip OK:", rebuilt.address.city == cust.address.city)

UK
36
round trip OK: True


## Summary

- A schema is the contract between messy generation and strict software; design it
  before the prompt, because its names and descriptions ARE part of the prompt.
- Construction validates: good data becomes typed objects; bad data raises one
  `ValidationError` carrying every finding in `.errors()`.
- Nest models to mirror real payloads; plain dicts coerce recursively at the boundary.
- `@field_validator` adds normalization and business rules beyond plain types.
- `model_dump` / `model_dump_json` / `model_validate_json` give lossless,
  revalidating round trips for storage and transport.

Next up: notebook 02 points REAL chat models at contracts like these - first via
`with_structured_output`, then via `PydanticOutputParser`, then fully offline.